<table style="width: 100%;">
<tr>
<td style="width: 50%; text-align: right; vertical-align: middle;">
<img src="https://github.com/gitpizzanow/dummy-files/blob/main/tp3nlp.jpg?raw=true" width="150">
</td>
<td style="width: 50%; text-align: left; vertical-align: middle;">

##  (NLP)   | TF-IDF + Cosine Similarity
> [SERIE](https://tp3-nlp-ing4.netlify.app/)


* *Document Frequency (DF)*
* *IDF + Smoothing*
* *TF (Term Frequency)*
* *Cosine Similarity*



</td>
</tr>
</table>

>Data: Wikipedia-like dataset

In [1]:
from sklearn.datasets import fetch_20newsgroups

docs = fetch_20newsgroups(
    subset='train',
    remove=('headers', 'footers', 'quotes')
).data[:3000]

In [2]:
type(docs)

list

In [3]:
docs[10]

'I have a line on a Ducati 900GTS 1978 model with 17k on the clock.  Runs\nvery well, paint is the bronze/brown/orange faded out, leaks a bit of oil\nand pops out of 1st with hard accel.  The shop will fix trans and oil \nleak.  They sold the bike to the 1 and only owner.  They want $3495, and\nI am thinking more like $3K.  Any opinions out there?  Please email me.\nThanks.  It would be a nice stable mate to the Beemer.  Then I\'ll get\na jap bike and call myself Axis Motors!\n\n-- \n-----------------------------------------------------------------------\n"Tuba" (Irwin)      "I honk therefore I am"     CompuTrac-Richardson,Tx\nirwin@cmptrc.lonestar.org    DoD #0826          (R75/6)'

![STEP 1 - Preprocessing](https://img.shields.io/badge/STEP%201%20-%20Preprocessing-blue)

In [4]:
#comlete the code
def preprocess(text):
    STOPWORDS = {
        "the","a","an","is","it","in","on","at","to","of","and","or","but",
        "for","with","as","by","that","this","was","are","be","have","has",
        "had","not","from","they","we","he","she","i","you","do","did","will",
        "would","could","should","its","their","our","my","your","been","were",
        "there","so","if","about","up","out","when","what","which","who","how",
        "all","more","also","than","then","can","just","into","over","after",
        "re","s","t","don","doesn","didn","isn","wasn","aren","won",
    }
    text   = text.lower()
    text   = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    return [t for t in tokens if len(t) > 2 and t not in STOPWORDS]

![STEP 2 - Vocabulary](https://img.shields.io/badge/STEP%202%20-%20Vocabulary-blue)

In [5]:
#comlete the code
def build_vocab(docs):
    vocab_set = set()
    for doc in docs:
        vocab_set.update(preprocess(doc))
    return sorted(vocab_set)

[![STEP 3 DF](https://img.shields.io/badge/STEP_3_DF-Document_Frequency-pink)](https://digitalpro.dev)

In [6]:
def compute_df(docs, vocab):
    vocab_index = set(vocab)
    df = {term: 0 for term in vocab}
    for doc in docs:
        tokens_in_doc = set(preprocess(doc)) & vocab_index
        for term in tokens_in_doc:
            df[term] += 1
    return df

[![STEP 4 IDF](https://img.shields.io/badge/STEP_4_IDF-Inverse_Document_Frequency-pink)](https://digitalpro.dev)

In [7]:
import math
def compute_idf(df, N):
    return {term: math.log((N + 1) / (freq + 1)) + 1 for term, freq in df.items()}


[![STEP 5 TF Vector](https://img.shields.io/badge/STEP_5_TF-Term_Frequency_Vector-pink)](https://digitalpro.dev)

In [8]:
import numpy as np
from collections import Counter
def compute_tf(doc, vocab):
    tokens= preprocess(doc)
    total= len(tokens) if tokens else 1
    counts= Counter(tokens)
    return np.array([counts.get(t, 0) / total for t in vocab], dtype=np.float64)

[![STEP 6 TF-IDF Matrix](https://img.shields.io/badge/STEP_6_TFIDF-Build_TF_IDF_Matrix-pink)](https://digitalpro.dev)

In [9]:
def build_tfidf(docs):
    vocab= build_vocab(docs)
    N= len(docs)
    df = compute_df(docs, vocab)
    idf= compute_idf(df, N)
    idf_vec= np.array([idf[t] for t in vocab], dtype=np.float64)
    matrix = np.zeros((N, len(vocab)), dtype=np.float64)
    for i, doc in enumerate(docs):
        matrix[i] = compute_tf(doc, vocab) * idf_vec   
    return matrix, vocab


[![STEP 7 Cosine Similarity](https://img.shields.io/badge/STEP_7-Cosine_Similarity-pink)](https://digitalpro.dev)

In [10]:
def cosine(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return 0.0 if denom == 0 else float(np.dot(a, b) / denom)

[![STEP 8 Search Engine](https://img.shields.io/badge/STEP_8-Search_Engine_REAL_VERSION-orange)](https://digitalpro.dev)

In [ ]:
import numpy as np
def search(query, docs, top_k=5):
    tfidf_matrix, vocab = build_tfidf(docs)

    q_vec = np.zeros(len(vocab))
    q_words = preprocess(query)

    for i, w in enumerate(vocab):
        q_vec[i] = q_words.count(w)

    if np.sum(q_vec) > 0:
        q_vec = q_vec / np.sum(q_vec)

    scores = []

    for i, doc_vec in enumerate(tfidf_matrix):
        score = cosine(q_vec, doc_vec)
        scores.append((score, i))

    scores.sort(reverse=True)

    return scores[:top_k]

> TEST

In [14]:
import re
query = "machine learning neural network"
results = search(query, docs)

for score, idx in results:
    print(score)
    print(docs[idx][:200])
    print("-" * 50)

0.2141256858904961
I just recently bought a 4 MB ram card for my original mac portable 
(backlit) and have since had some bizarre crashes. It happens when I put 
the machine to sleep and wake the machine up. sometimes i
--------------------------------------------------
0.1860847834113687
OK all you experts!
Need answer quick.386 machine ,1.44 floppy ; unable to write to a formated
720 disk.Machine claims that disk is write protected,but it is not.

Note: It 'll read 720's with no prob
--------------------------------------------------
0.17552464652505897
I'm looking for a Singer Featherweight 221 sewing machine (old, black 
sewing machine in black case).

Please contact:
--------------------------------------------------
0.15516665492150827



The previous article referred to the fact that you could only use 20ns SIMMs in
a 50MHz machine, but that you could use 80ns SIMMs in slower machines. I just
pointed out that if you could only use 
-----------------------------------------------